# Second-order methods

The previous lessons are a great foundation in numerical methods for studying dynamical systems governed by ordinary differential equations.
You learned to apply Euler's method and studied its rate of convergence using numerical experiments. An exercise **on paper** in [Lesson 2](./02-oscillation.ipynb) using Taylor expansions showed that convergence to be first order. We confirmed this behavior in [Lesson 3](./03-full-model.ipynb) using the full nonlinear phugoid model.

We did hint that Euler's method has some limitations: for oscillatory systems every Euler step enlarges the oscillation slightly. Many problems also require a more accurate method, second order or higher. Among the most popular higher-order methods are the _Runge-Kutta methods_, developed around 1900: more than 100 years after Euler published his book containing the method now named after him.

In a first-order method, the error scales _linearly_ with the step size:

$$
e \propto \Delta t
$$

where $e$ stands for the error. In a _second-order_ method, the error is ${\mathcal O}(\Delta t^2)$. In general, we say that a method is of order $p$ when the error is proportional to $(\Delta t)^p$.

One idea for improving on Euler's method is to estimate the derivative at an intermediate point, like the **midpoint**, which results in the so-called *explicit midpoint method* or *modified Euler method*. The scheme has two steps and is written as:

$$
\label{eq-rk2-midpoint-system}
\begin{aligned}
u_{n+1/2}   & = u_n + \frac{\Delta t}{2} f(u_n) \\
u_{n+1} & = u_n + \Delta t \,\, f(u_{n+1/2})
\end{aligned}
$$

Notice that we had to apply the right-hand side, $~f(u)$, twice. This idea can be extended: we could imagine estimating additional points between $u_{n}$ and $u_{n+1}$ and evaluating $~f(u)$ at the intermediate points to get higher accuracy—that's the idea behind Runge-Kutta methods.

## Runge–Kutta methods

In the modified Euler method, we improve the accuracy over Euler's method by evaluating the right-hand side of the differential equation at an intermediate point: the midpoint. The same idea can be applied again, and the function $f(u)$ can be evaluated at more intermediate points, improving the accuracy even more. This is the basis of the famous *Runge-Kutta (RK) methods*, going back to Carl Runge and Martin Kutta. The modified Euler method corresponds to _second-order_ Runge-Kutta.

There is a historical connection to our phugoid problem: Carl Runge's daughter Iris—an accomplished applied mathematician in her own right—worked assiduously over the summer of 1909 to translate Lanchester's _"Aerodonetics."_ She also reproduced his graphical method to draw the phugoid curves [@tobies2012, p. 73].

## Phugoid model with second-order RK

Let's compute the motion of a glider under the full phugoid model using the second-order Runge–Kutta method. This calculation sets the stage for the paper-airplane challenge later in this lesson: predicting how far a paper airplane flies before touching the ground.

Start by importing NumPy and Matplotlib with the aliases used in the previous lessons. We also set the font family and size through Matplotlib's `rcParams` dictionary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the font family and size to use for Matplotlib figures.
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

For our paper-airplane model, we will use $L/D=5.0$, motivated by experimental measurements [@feng2009], and a trim speed of $4.9\ \mathrm{m/s}$. _What do you think will happen if you make $L/D$ higher?_

In [ ]:
# Model parameters.
g = 9.81  # gravitational acceleration (m/s**2)
v_t = 4.9  # trim speed (m/s)
C_D = 1.0 / 5.0  # drag coefficient
C_L = 1.0  # lift coefficient

# Initial conditions.
v_0 = 6.5  # initial speed, above the trim speed (m/s)
theta_0 = -0.1  # trajectory angle (rad)
x_0 = 0.0  # horizontal position (m)
y_0 = 2.0  # altitude (m)

The initial speed is a little higher than the trim speed, the launch angle is negative, and the release height is 2 meters. We will use the same initial state and parameters for both numerical methods.

### Reuse functions from a Python module

We have already written and inspected three functions:

- `rhs_full_phugoid()` gives the four model derivatives, as developed in [Lesson 3](./03-full-model.ipynb).
- `euler_step()` advances the complete state by one Forward Euler step, using the pattern introduced in [Lesson 2](./02-oscillation.ipynb).
- `discrete_l1_difference()` compares histories on nested time grids, as developed in [Lesson 3](./03-full-model.ipynb).

These functions are now reused often enough to save in a separate file. A **module** is an ordinary Python source file with the extension `.py`. To make one, create a plain-text file, copy the function definitions into it, and include the imports those definitions need—for these functions, `import numpy as np`. Keep parameter choices, integration loops, plots, and notebook-only commands in the notebook. Saving a function does not run a simulation.

The course provides these definitions in a file named `phugoid.py`. [Read the module source](https://github.com/numerical-mooc/practical-numerical-methods/blob/main/src/phugoid.py): its three implementations are the ones already presented in Lesson 3. We can download this file directly using `urlretrieve()` from Python's standard library. No installation of course code is needed.

In [ ]:
from urllib.request import urlretrieve

url = (
    'https://raw.githubusercontent.com/'
    'numerical-mooc/practical-numerical-methods/main/'
    'src/phugoid.py'
)
fname = 'phugoid.py'
urlretrieve(url, fname)

Here, `url` points to the raw Python file, and `fname` names the local copy. The file is saved in the kernel's current working directory, normally the folder containing your notebook. Keep `phugoid.py` alongside your working notebook. You only need to download it once; rerunning the download overwrites that file, so save a separate copy of any local edits first.

Downloading saves the file; **importing** makes its functions available. In the import statement, use the filename without `.py`. Only import code from sources you trust and have inspected: Python executes a module's top-level statements when it first loads it. Our file imports NumPy and defines functions; it does not run a simulation.

Functions in the module do not inherit notebook variables, so we continue to pass the state and model parameters explicitly. The difference function requires **both** time-step sizes to check grid nesting and endpoint alignment.

If you edit the local module, restart the kernel and rerun your imports and calculations, skipping the download cell to preserve your edits. Rerunning an import alone does not reload an already imported module. For more detail, see the [Python modules tutorial](https://docs.python.org/3/tutorial/modules.html).

In [ ]:
from phugoid import (
    discrete_l1_difference,
    euler_step,
    rhs_full_phugoid,
)

### Define the RK2 step

The reused functions now come from the module, but the new numerical method stays visible here. Define `rk2_step()` to implement the modified Euler method in [Equation %s](#eq-rk2-midpoint-system), also known as second-order Runge–Kutta or RK2. The time loop will call this function once per step.

In [ ]:
def rk2_step(u, f, dt, *args):
    '''Return the next state using the second-order Runge–Kutta method.

    Parameters
    ----------
    u : np.ndarray
        State at the current time
        as a 1D array of floats.
    f : function
        Function to compute the right-hand side of the system.
    dt : float
        Time-step size.
    *args
        Additional positional arguments passed to f.

    Returns
    -------
    u_new : np.ndarray
        The solution at the next time step
        as a 1D array of floats.
    '''
    u_star = u + 0.5 * dt * f(u, *args)
    u_new = u + dt * f(u_star, *args)
    return u_new

### Integrate with both methods

As in [Lesson 3](./03-full-model.ipynb), choose the final time and time-step size, then allocate the state histories. The variable `num_steps` counts updates; each history has `num_steps + 1` rows to include the initial state. Both arrays have four columns ordered as `[v, theta, x, y]`. The same loop advances the two histories, once with Euler's method and once with RK2.

In [ ]:
T = 15.0  # length of the time interval (s)
dt = 0.01  # time-step size (s)
num_steps = int(round(T / dt))

# Store the state at every time point, including the initial state.
u_euler = np.empty((num_steps + 1, 4))
u_rk2 = np.empty((num_steps + 1, 4))
u_euler[0] = np.array([v_0, theta_0, x_0, y_0])
u_rk2[0] = np.array([v_0, theta_0, x_0, y_0])

# Advance both methods over the same time grid.
for n in range(num_steps):
    u_euler[n + 1] = euler_step(
        u_euler[n], rhs_full_phugoid, dt, C_L, C_D, g, v_t
    )
    u_rk2[n + 1] = rk2_step(
        u_rk2[n], rhs_full_phugoid, dt, C_L, C_D, g, v_t
    )

Extract the horizontal position and altitude from columns 2 and 3 of each state history, using the same array slicing as in [Lesson 3](./03-full-model.ipynb).

In [ ]:
# Extract the position history for each method.
x_euler = u_euler[:, 2]
y_euler = u_euler[:, 3]
x_rk2 = u_rk2[:, 2]
y_rk2 = u_rk2[:, 3]

### How far will it fly before touching the ground?

As the $y$-axis measures the vertical coordinate with respect to the ground, negative values of $y$ don't have any physical meaning: the glider would have hit the ground by then! To find out if there are any negative $y$ values we can use the handy function [`np.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html). This function returns the **indices** of the elements in an array that match a given condition. For example, `np.where(y_euler<0)[0]` gives an array of the indices `i` where `y_euler[i]<0` (`np.where` returns a tuple with one index array per dimension; `[0]` selects the array for this one-dimensional history). If no elements of the array match the condition, the array of indices comes out empty.

From the physical problem, we know that once there is one negative value, the glider has hit the ground and all the remaining time-steps are unphysical. Therefore, we are interested in finding the _first_ index where the condition applies, given by `np.where(y_euler<0)[0][0]`—do read the  documentation of the function if you need to!

In [ ]:
# Get the index of the first negative element of y_euler.
idx_negative_euler = np.where(y_euler < 0.0)[0]
if len(idx_negative_euler) == 0:
    idx_ground_euler = num_steps
    print('[Euler] Glider has not touched ground yet!')
else:
    idx_ground_euler = idx_negative_euler[0]

# Get the index of the first negative element of y_rk2.
idx_negative_rk2 = np.where(y_rk2 < 0.0)[0]
if len(idx_negative_rk2) == 0:
    idx_ground_rk2 = num_steps
    print('[RK2] Glider has not touched ground yet!')
else:
    idx_ground_rk2 = idx_negative_rk2[0]

### Do Euler and RK2 produce the same solution?

An easy way to compare the numerical results obtained with the Euler and 2nd-order Runge-Kutta methods is using [`np.allclose`](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html). This function compares each element of two arrays and returns `True` if each comparison is within some relative tolerance. Here, we use the default tolerance: $10^{-5}$.

In [ ]:
# Compare the two position histories.
print(f'Are the x-values close? {np.allclose(x_euler, x_rk2)}')
print(f'Are the y-values close? {np.allclose(y_euler, y_rk2)}')

The arrays differ at the chosen tolerance. Maybe $10^{-5}$ is too tight a tolerance, considering we're using a somewhat coarse grid with first- and second-order methods. Perhaps we can assess this visually, by plotting the glider's path? Study the code below, where we are plotting the path twice, taking a closer look in the second plot by zooming in on the beginning of the flight.

In [ ]:
print(f'Distance traveled: {x_rk2[idx_ground_rk2 - 1]:.3f} m')

fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0))
for ax in axes:
    ax.grid()
    ax.set_xlabel('Horizontal position, x (m)')
    ax.set_ylabel('Altitude, y (m)')

# Plot the flight path for both methods.
axes[0].plot(
    x_euler[:idx_ground_euler], y_euler[:idx_ground_euler],
    label='Euler',
)
axes[0].plot(
    x_rk2[:idx_ground_rk2], y_rk2[:idx_ground_rk2], label='RK2'
)
axes[0].legend()

# Zoom in on the beginning of the flight.
axes[1].plot(x_euler, y_euler, label='Euler')
axes[1].plot(x_rk2, y_rk2, label='RK2')
axes[1].set_xlim(0.0, 5.0)
axes[1].set_ylim(1.8, 2.5)
fig.tight_layout()

From far away, the Euler and RK2 methods seem to be producing similar answers. However, if we take a closer look, small differences become evident. Keep in mind that we are solving the same equation and both methods will converge to the same solution as we refine the grid. However, they converge to that solution at different rates: RK2 gets more accurate faster, as you make $\Delta t$ smaller.

## Grid convergence

Just like in [Lesson 3](./03-full-model.ipynb), we want to do a grid-convergence study with RK2, to see if we indeed observe the expected rate of convergence. It is always an important step in a numerical solution to investigate whether the method is behaving the way we expect it to: this needs to be confirmed experimentally for every new problem we solve and for every new method we apply!

In the code below, a `for`-loop computes the solution on different time grids, with the coarsest and finest grid differing by 100x. We can use the difference between solutions to investigate convergence, as before.

In [ ]:
# Set the time-step sizes to investigate.
dt_values = [0.1, 0.05, 0.01, 0.005, 0.001]
u_histories = []

for dt_trial in dt_values:
    num_steps_trial = int(round(T / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    for n in range(num_steps_trial):
        u_trial[n + 1] = rk2_step(
            u_trial[n], rhs_full_phugoid, dt_trial,
            C_L, C_D, g, v_t,
        )
    u_histories.append(u_trial)

Once those runs are complete, compare each horizontal-position history with the finest-grid history. Pass both time-step sizes to the imported `discrete_l1_difference()` function so it can align the grids.

In [ ]:
# Compute the differences in horizontal position.
difference_values = []
for u_trial, dt_trial in zip(u_histories, dt_values, strict=True):
    difference = discrete_l1_difference(
        u_trial[:, 2], u_histories[-1][:, 2],
        dt_trial, dt_values[-1],
    )
    difference_values.append(difference)

Plot the differences against time-step size on logarithmic axes, as in the previous lessons.

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.set_title(r'$L_1$ difference vs. time-step size')
ax.set_xlabel(r'$\Delta t$ (s)')
ax.set_ylabel(r'$L_1$ difference in $x$')
ax.grid()
ax.loglog(
    dt_values[:-1], difference_values[:-1],
    color='tab:blue', linestyle='--', marker='o',
)
ax.set_aspect('equal', adjustable='box')
fig.tight_layout()

The difference relative to our fine-grid solution is decreasing with the mesh size at a faster rate than in [Lesson 3](./03-full-model.ipynb), but *how much faster?* When we computed the observed order of convergence with Euler's method, we got a value close to 1—it's a first-order method. Can you guess what we'll get now with RK2?

To compute the observed order of convergence, we use three grid resolutions that are refined at a constant rate, in this case $r=2$.

In [ ]:
refinement_ratio = 2
dt_fine = 0.001
dt_order = [
    dt_fine,
    refinement_ratio * dt_fine,
    refinement_ratio**2 * dt_fine,
]
u_order = []

for dt_trial in dt_order:
    num_steps_trial = int(round(T / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    for n in range(num_steps_trial):
        u_trial[n + 1] = rk2_step(
            u_trial[n], rhs_full_phugoid, dt_trial,
            C_L, C_D, g, v_t,
        )
    u_order.append(u_trial)

# Compute the observed order of convergence.
observed_order = np.log(
    discrete_l1_difference(
        u_order[2], u_order[1], dt_order[2], dt_order[1]
    )
    / discrete_l1_difference(
        u_order[1], u_order[0], dt_order[1], dt_order[0]
    )
) / np.log(refinement_ratio)
print(f'Observed order of convergence: p = {observed_order:.3f}')

Probably you're not too surprised to see that the observed order of convergence is close to $2$. Because we used a second-order method! This means that the numerical solution is converging with the grid resolution twice as fast compared with Euler's method in [Lesson 3](./03-full-model.ipynb), or in other words, the error scales as ${\mathcal O}(\Delta t^2)$. That is a lot faster! However, we are paying a price here: second-order Runge-Kutta requires more computations per iteration.

### Compare runtimes

How much longer does it take to get the solution with RK2, compared to Euler's method? Run the same calculation (same time grid, same parameters), but find a way to *time* the calculation with Python, and compare the runtimes.

## Multi-step methods

The midpoint method introduced an intermediate state between $u_n$ and $u_{n+1}$ and evaluated the right-hand side there. Multi-step methods draw on a different source of information: states from earlier time steps.

For example, we can involve in the calculation of the solution $u_{n+1}$ the known solution at $u_{n-1}$, in addition to $u_{n}$. Schemes that use this idea are called _multi-step methods_.

A classical multi-step method achieves second order by applying a _centered difference_ approximation of the derivative $u'$:

$$
\label{eq-centered-time-derivative}
u'(t) \approx \frac{u_{n+1} - u_{n-1}}{2\Delta t}
$$

Isolate the future value of the solution $u_{n+1}$ and apply the differential equation $u'=f(u)$, to get the following formula for this method:

$$
\label{eq-leapfrog-update}
u_{n+1} = u_{n-1} + 2\Delta t \, f(u_n)
$$

This scheme is known as the **leapfrog method**. Notice that it is using the right-hand side of the differential equation, $f(u)$, evaluated at the _midpoint_ between $u_{n-1}$ and $u_{n+1}$, where the time interval between these two solutions is $2\Delta t$. Why is it called "leapfrog"? If you imagine for a moment all of the _even_ indices $n$ of the numerical solution, you notice that these solution values are computed using the slope estimated from _odd_ values $n$, and vice-versa.

Let's define a function that computes the numerical solution using the leapfrog method:

In [ ]:
def leapfrog_step(u_prev, u, f, dt, *args):
    '''Return the next state using the leapfrog method.

    Parameters
    ----------
    u_prev : np.ndarray
        Solution at the time step n-1
        as a 1D array of floats.
    u : np.ndarray
        Solution at the previous time step
        as a 1D array of floats.
    f : function
        Function to compute the right-hand side of the system.
    dt : float
        Time-step size.
    *args
        Additional positional arguments passed to f.

    Returns
    -------
    u_new : np.ndarray
        The solution at the next time step
        as a 1D array of floats.
    '''
    u_new = u_prev + 2.0 * dt * f(u, *args)
    return u_new

But wait ... what will we do at the _initial_ time step, when we don't have information for $u_{n-1}$? This is an issue with all multi-step methods: we say that they are _not self-starting_. In the first time step, we need to use another method to get the first "kick"—either Euler's method or 2nd-order Runge Kutta could do: let's use RK2, since it's also second order.

For this calculation, we are going to re-enter the model parameters in the code cell below, so that later on we can experiment here using the leapfrog method and different starting values. At the end of this notebook, we'll give you some other model parameters to try that will create a very interesting situation!

In [ ]:
# Model parameters.
g = 9.81  # gravitational acceleration (m/s**2)
v_t = 4.9  # trim speed (m/s)
C_D = 1.0 / 5.0  # drag coefficient
C_L = 1.0  # lift coefficient

# Initial conditions.
v_0 = 6.5  # initial speed, above the trim speed (m/s)
theta_0 = -0.1  # trajectory angle (rad)
x_0 = 0.0  # horizontal position (m)
y_0 = 2.0  # altitude (m)

T = 15.0  # length of the time interval (s)
dt = 0.01  # time-step size (s)
num_steps = int(round(T / dt))

u_leapfrog = np.empty((num_steps + 1, 4))
u_leapfrog[0] = np.array([v_0, theta_0, x_0, y_0])

# Use RK2 to obtain the second state before starting leapfrog.
u_leapfrog[1] = rk2_step(
    u_leapfrog[0], rhs_full_phugoid, dt, C_L, C_D, g, v_t
)

Now we have all the required information to loop in time using the leapfrog method. The code cell below calls the leapfrog function for each time step.

In [ ]:
# Advance with leapfrog once the first two states are available.
for n in range(1, num_steps):
    u_leapfrog[n + 1] = leapfrog_step(
        u_leapfrog[n - 1], u_leapfrog[n], rhs_full_phugoid, dt,
        C_L, C_D, g, v_t,
    )

Like before, we extract from the solution array the information about the glider's position in time and find where it reaches the ground.

In [ ]:
# Extract the position history.
x_leapfrog = u_leapfrog[:, 2]
y_leapfrog = u_leapfrog[:, 3]

# Get the index of the first negative element of y_leapfrog.
idx_negative_leapfrog = np.where(y_leapfrog < 0.0)[0]
if len(idx_negative_leapfrog) == 0:
    idx_ground_leapfrog = num_steps
    print('[leapfrog] Glider has not touched ground yet!')
else:
    idx_ground_leapfrog = idx_negative_leapfrog[0]

Plotting the glider's trajectory with both the leapfrog and RK2 methods, we find that the solutions are very close to each other now: we don't see the differences that were apparent when we compared Euler's method and RK2.

In [ ]:
print(f'Distance traveled: {x_leapfrog[idx_ground_leapfrog - 1]:.3f} m')

fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0))
for ax in axes:
    ax.grid()
    ax.set_xlabel('Horizontal position, x (m)')
    ax.set_ylabel('Altitude, y (m)')

# Plot the flight path computed with leapfrog.
axes[0].plot(
    x_leapfrog[:idx_ground_leapfrog],
    y_leapfrog[:idx_ground_leapfrog],
)

# Zoom in on the beginning of the flight.
axes[1].plot(x_leapfrog, y_leapfrog)
axes[1].set_xlim(0.0, 5.0)
axes[1].set_ylim(1.8, 2.5)
fig.tight_layout()

What about the observed order of convergence? We'll repeat the process we have used before, with a grid-refinement ratio $r=2$.

In [ ]:
refinement_ratio = 2
dt_fine = 0.001
dt_order = [
    dt_fine,
    refinement_ratio * dt_fine,
    refinement_ratio**2 * dt_fine,
]
u_order = []

for dt_trial in dt_order:
    num_steps_trial = int(round(T / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    # Use RK2 for the first time step.
    u_trial[1] = rk2_step(
        u_trial[0], rhs_full_phugoid, dt_trial, C_L, C_D, g, v_t
    )
    for n in range(1, num_steps_trial):
        u_trial[n + 1] = leapfrog_step(
            u_trial[n - 1], u_trial[n], rhs_full_phugoid, dt_trial,
            C_L, C_D, g, v_t,
        )
    u_order.append(u_trial)

# Compute the observed order of convergence.
observed_order = np.log(
    discrete_l1_difference(
        u_order[2][:, 2], u_order[1][:, 2],
        dt_order[2], dt_order[1],
    )
    / discrete_l1_difference(
        u_order[1][:, 2], u_order[0][:, 2],
        dt_order[1], dt_order[0],
    )
) / np.log(refinement_ratio)
print(f'Observed order of convergence: p = {observed_order:.3f}')

We now have numerical evidence that our calculation with the leapfrog method indeed exhibits second-order convergence, i.e., the method is ${\mathcal O}(\Delta t^2)$. _The leapfrog method is a second-order method_. Good job!

### A longer flight

Go back to the cell that re-enters the model parameters, just above the leapfrog-method time loop, and change the following: the initial height `y_0` to 25, and the final time `T` to 36. Now re-run the leapfrog calculation and the two code cells below that, which extract the glider's position and plot it.

_What is going on?_